In [1]:
import pandas as pd
import optuna
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score
from imblearn.over_sampling import SMOTE
from imblearn.over_sampling import SMOTE

In [2]:
# Load the dataset
df = pd.read_csv(r'C:\Users\khiew\Downloads\FYP Reduced Dataset.csv')

In [3]:
# Drop diseases with less than 50 instances
disease_counts = df['diseases'].value_counts()
valid_diseases = disease_counts[disease_counts >= 500].index
df = df[df['diseases'].isin(valid_diseases)]

# Assuming that the target variable is 'diseases' and all other variables are input features
X = df.drop('diseases', axis=1)
y = df['diseases']

# Encode the target variable (diseases) if it's a categorical variable
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)
print("Number of remaining classes in training set:", len(np.unique(y_train)))
print("Number of rows left:", len(df))

Number of remaining classes in training set: 201
Number of rows left: 168499


In [4]:
# Apply SMOTE for class balancing in the training set
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# Print the new class distribution after SMOTE
print("Class distribution after SMOTE:")
print(pd.Series(y_train_resampled).value_counts())
print("Number of remaining classes in training set:", len(np.unique(y_train_resampled)))
# Print the number of rows in the resampled training set
print("Number of rows in the resampled training set:", len(X_train_resampled))

Class distribution after SMOTE:
88     1002
150    1002
123    1002
125    1002
141    1002
       ... 
108    1002
39     1002
96     1002
192    1002
118    1002
Name: count, Length: 201, dtype: int64
Number of remaining classes in training set: 201
Number of rows in the resampled training set: 201402


In [6]:
# Optuna optimization function
def objective(trial):
    # Define the hyperparameters to tune
    n_estimators = trial.suggest_int('n_estimators', 50, 150)
    max_depth = trial.suggest_int('max_depth', 10, 50)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 20)
    max_features = trial.suggest_categorical('max_features', ['sqrt', 'log2', None])

    # Create RandomForestClassifier with hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        random_state=42
    )

    # Perform cross-validation (using 5-fold by default)
    score = cross_val_score(model, X_train_resampled, y_train_resampled, cv=5, scoring='accuracy')
    accuracy = score.mean()

    # Log the hyperparameters and accuracy for this trial
    print(f"Trial {trial.number}: n_estimators={n_estimators}, max_depth={max_depth}, "
          f"min_samples_split={min_samples_split}, min_samples_leaf={min_samples_leaf}, "
          f"max_features={max_features}, Accuracy={accuracy:.4f}")

    return accuracy

In [7]:
# Create Optuna study for optimization with persistent storage
study = optuna.create_study(
    direction='maximize',  # Assuming you are maximizing accuracy
    study_name="randomforest_diseases_symptoms_dropextremelymore500+SMOTE_study", 
    storage=r"sqlite:///C:/Users/khiew/Downloads/randomforest.db", 
    load_if_exists=True  # Load the study if it already exists, to resume from the last trial
)

# Optimize the study with your objective function, you can adjust the n_trials as needed
study.optimize(objective, n_trials=20)
# Print the best trial and hyperparameters
print("\nBest Trial:")
print(study.best_trial)
print("Best Hyperparameters:")
print(study.best_trial.params)

[I 2025-04-22 17:08:26,829] A new study created in RDB with name: randomforest_diseases_symptoms_dropextremelymore500+SMOTE_study
[I 2025-04-22 17:09:08,398] Trial 0 finished with value: 0.39978252293526756 and parameters: {'n_estimators': 88, 'max_depth': 37, 'min_samples_split': 2, 'min_samples_leaf': 13, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.39978252293526756.


Trial 0: n_estimators=88, max_depth=37, min_samples_split=2, min_samples_leaf=13, max_features=sqrt, Accuracy=0.3998


[I 2025-04-22 17:10:07,512] Trial 1 finished with value: 0.400159880932773 and parameters: {'n_estimators': 124, 'max_depth': 40, 'min_samples_split': 18, 'min_samples_leaf': 14, 'max_features': 'sqrt'}. Best is trial 1 with value: 0.400159880932773.


Trial 1: n_estimators=124, max_depth=40, min_samples_split=18, min_samples_leaf=14, max_features=sqrt, Accuracy=0.4002


[I 2025-04-22 17:10:54,904] Trial 2 finished with value: 0.40012513076274725 and parameters: {'n_estimators': 90, 'max_depth': 29, 'min_samples_split': 9, 'min_samples_leaf': 2, 'max_features': 'sqrt'}. Best is trial 1 with value: 0.400159880932773.


...

Trial 17: n_estimators=51, max_depth=23, min_samples_split=16, min_samples_leaf=10, max_features=log2, Accuracy=0.3998


[I 2025-04-22 17:28:32,669] Trial 18 finished with value: 0.3999960354231484 and parameters: {'n_estimators': 110, 'max_depth': 32, 'min_samples_split': 18, 'min_samples_leaf': 5, 'max_features': 'log2'}. Best is trial 12 with value: 0.40051241382318015.


Trial 18: n_estimators=110, max_depth=32, min_samples_split=18, min_samples_leaf=5, max_features=log2, Accuracy=0.4000


[I 2025-04-22 17:29:27,164] Trial 19 finished with value: 0.4004280022218296 and parameters: {'n_estimators': 133, 'max_depth': 27, 'min_samples_split': 13, 'min_samples_leaf': 12, 'max_features': 'log2'}. Best is trial 12 with value: 0.40051241382318015.


Trial 19: n_estimators=133, max_depth=27, min_samples_split=13, min_samples_leaf=12, max_features=log2, Accuracy=0.4004

Best Trial:
FrozenTrial(number=12, state=TrialState.COMPLETE, values=[0.40051241382318015], datetime_start=datetime.datetime(2025, 4, 22, 17, 22, 58, 263453), datetime_complete=datetime.datetime(2025, 4, 22, 17, 23, 47, 980242), params={'n_estimators': 114, 'max_depth': 34, 'min_samples_split': 20, 'min_samples_leaf': 8, 'max_features': 'log2'}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=150, log=False, low=50, step=1), 'max_depth': IntDistribution(high=50, log=False, low=10, step=1), 'min_samples_split': IntDistribution(high=20, log=False, low=2, step=1), 'min_samples_leaf': IntDistribution(high=20, log=False, low=1, step=1), 'max_features': CategoricalDistribution(choices=('sqrt', 'log2', None))}, trial_id=233, value=None)
Best Hyperparameters:
{'n_estimators': 114, 'max_depth': 34, 'min_samples_split